## MongoDB Query Workbook
`$cluster_name` - `$database_name.$collection_name` - `$query_hash`

In [ ]:
# Initialize

# Imports
import json
import os

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
from IPython.core.display import Markdown
from IPython.display import display
from anthropic import Anthropic
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from pymongo.mongo_client import MongoClient

# Load env variables for
# - MongoDB Atlas CLI,
# - MongoDB Client
# - Anthropic client
load_dotenv()

cluster_name = "$cluster_name"
database_name = "$database_name"
collection_name = "$collection_name"
query_hash = "$query_hash"

# Create MongoBD Client, point to a specific database/collection
uri = os.getenv("MONGO_URI")
mongodb_client = MongoClient(uri)
namespace = database_name + "." + collection_name
db = mongodb_client[database_name][collection_name]
stats_db = mongodb_client.get_database("observability").get_collection("queryStats")

# Create Anthropic client
anthropic_client = Anthropic()
base_context = []


# MongoDB context

def webpage_as_context(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    body = soup.select('div.body')[0]
    text = body.get_text(separator=" ", strip=True)
    return {
        "role": "user",
        "content": f"MongoDB documentation from {url}\n{text}",
    }

docs = [
    "https://www.mongodb.com/docs/manual/core/query-optimization/",
    "https://www.mongodb.com/docs/manual/tutorial/equality-sort-range-rule/",
    "https://www.mongodb.com/docs/manual/core/index-partial/",
    "https://www.mongodb.com/docs/manual/reference/explain-results/",
    "https://www.mongodb.com/docs/manual/tutorial/analyze-query-plan/",
    "https://www.mongodb.com/docs/manual/tutorial/optimize-query-performance-with-indexes-and-projections/",
    "https://www.mongodb.com/docs/atlas/performance-advisor/",
    "https://www.mongodb.com/docs/atlas/performance-advisor/index-ranking/",
    "https://www.mongodb.com/docs/manual/reference/operator/aggregation/indexStats/",
    "https://www.mongodb.com/docs/cloud-manager/reference/api/performance-advisor/get-suggested-indexes/",
]

for url in docs:
    context_message = webpage_as_context(url)
    base_context.append(context_message)

# Current indexes
# https://www.mongodb.com/docs/manual/reference/operator/aggregation/indexStats/
current_indexes = []

index_stats = list(db.aggregate([{"$indexStats": {}}]))
for idx in index_stats:
    current_indexes.append(str(idx["key"]))
current_indexes = sorted(current_indexes)

# Suggested indexes
# https://www.mongodb.com/docs/atlas/cli/current/command/atlas-api-performanceAdvisor-listClusterSuggestedIndexes/
suggested_indexes_details = []

index_suggestions_raw = !atlas api performanceAdvisor listClusterSuggestedIndexes --clusterName {cluster_name} --output json --version "2024-08-05" --namespaces {namespace}
index_suggestions = json.loads(index_suggestions_raw.s)

for suggestion in index_suggestions["content"]["suggestedIndexes"]:

    # Convert to json string index definition
    index_definition = {}
    for record in suggestion["index"]:
        index_definition.update(record)

    index_definition = str(index_definition)

    for sample_shape in index_suggestions["content"]["shapes"]:
        if sample_shape["id"] in suggestion["impact"]:

            # Use first sample query
            operation = sample_shape["operations"][0]

            operation["stats"].update({"count": sample_shape["count"]})
            if "ts" in operation["stats"]:
                del operation["stats"]["ts"]

            if "raw" in operation:
                sample_query_raw = json.loads(operation["raw"])
                qh = "#" + sample_query_raw["attr"]["queryShapeHash"][0:6]

                if query_hash == qh:
                    suggested_indexes_details.append({
                        "suggested index": index_definition,
                        "query hash": qh,
                    })

# Get historical stats
query_stats_history = list(stats_db.aggregate([
    {"$match": {
        "db": database_name,
        "coll": collection_name,
        "hash": query_hash
    }},
    {"$project": {
        "_id": 0,
    }},
]))

df_history = pd.DataFrame.from_dict(query_stats_history)
df_history['timestamp'] = pd.to_datetime(df_history['timestamp'])

#### <span style="color:$query_hash">$query_hash</span> $query_short_description
$query_details

In [ ]:
$warnings
query = $query_example
#TODO replace ?value placeholders with realistic data to evaluate the query

### Query performance

In [ ]:
# Historical stats
fig = px.scatter(
    df_history,
    x="timestamp",
    y="avg",
    error_y="err+",
    error_y_minus="err-",
    facet_row="hash",
    color="hash",
    color_discrete_sequence=df_history["hash"],
    # log_x=True,
    height=300,
)
# fig.update_xaxes(matches=None)
fig.update_yaxes(matches=None)

fig.update_layout(xaxis_title="Query execution time, ms")
fig.update_traces(marker_size=8)
fig.for_each_xaxis(lambda xaxis: xaxis.update(showticklabels=True))
for axis in fig.layout:
    if type(fig.layout[axis]) == go.layout.YAxis:
        fig.layout[axis].title.text = ''

fig.show()

In [ ]:
# Measure end-to-end execution time
time = %timeit -o list(db.aggregate(pipeline=query) if type(query) is list else list(query.clone()))

In [ ]:
# Explain query
# https://www.mongodb.com/docs/manual/reference/explain-results/
if type(query) is list:
    explain = mongodb_client.get_database(database_name).command(
        'explain',
        {
            'aggregate': collection_name,
            'pipeline': query,
            'cursor': {}
        },
        verbosity='allPlansExecution'
    )
else:
    explain = query.clone().explain()

# Generate summary
context = base_context.copy()
context.extend([
    {
        "role": "user",
        "content": f"Act as a MongoDB advisor. You will be provided with information on indexes, example query that is being evaluated and a query explain plan. Use text graphics to visualise Execution Stages in Query Explain. Provide recommendations for query performance and indexes. Use markdown format for output, escape $ $ character sequence in markdown if it's not in the code block. In the first section, include original query. Then in the first section print 'Query hash:' and only first 6 symbols of a query hash, starting with # (e.g: #ABCDEF) in text color, exactly matching the hash value with the html color # code.",
    },
    {
        "role": "user",
        "content": f"Current indexes: {current_indexes}, suggested indexes: {suggested_indexes_details}",
    },
    {
        "role": "user",
        "content": f"Index stats: {index_stats}",
    },
    {
        "role": "user",
        "content": f"Query Explain raw: {explain}",
    },
    {
        "role": "user",
        "content": f"Execution time on the client side: {time}",
    }
])
response = anthropic_client.messages.create(
    max_tokens=1024,
    messages=context,
    model="claude-3-7-sonnet-latest",
)

display(Markdown(response.content[0].text))
context.extend([
    {
        "role": "assistant",
        "content": response.content[0].text,
    }
])